In [ ]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [ ]:
import gensim
from gensim.models import Word2Vec
import pandas as pd
import nltk
import os
from nltk.tokenize import word_tokenize
import gensim.downloader as api
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
gendered1 = pd.read_csv('../data/letters_2021_processed_with_gender.csv')
gendered1 = gendered1[['s1_s2', 'LETTER_GENDER']]
gendered1 = gendered1.rename(columns={'LETTER_GENDER':'label'})

In [ ]:
gendered2 = pd.read_csv('../data/sentence_sets_trimmed_processed_with_gender.csv', encoding='mac-roman')
gendered2 = gendered2[['s1_s2', 'applicant_gender']]
gendered2 = gendered2.rename(columns={'TEXT':'LETTERTEXT', 'applicant_gender':'label'})

In [ ]:
gendered = pd.concat([gendered1, gendered2], ignore_index=True)

In [ ]:
gender_label_mapping = {
    'F':0,
    'female':0,
    'M':1,
    'male':1
}

In [ ]:
gendered['label'] = gendered['label'].replace(gender_label_mapping)

<ipython-input-13-d6a02b119c3b>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gendered['label'] = gendered['label'].replace(gender_label_mapping)


In [ ]:
gendered

,s1_s2,label
0,it is my pleasure to write a letter of recomme...,0
1,i am pleased to highly recommend identifier fo...,0
2,i am writing this letter in support of identif...,0
3,identifier identifier recently completed an an...,0
4,it is my pleasure to recommend dr. identifier ...,0
...,...,...
8982,it is with pleasure that i recommend first_nam...,0
8983,we are very pleased to write this letter based...,0
8984,| am writing this letter of recommendation for...,1
8985,it is my pleasure to write first_name support ...,1


In [ ]:
sentences = [word_tokenize(text.lower()) for text in gendered["s1_s2"]]

In [ ]:
model = Word2Vec(
    sentences,          # Your tokenized sentences
    vector_size=50,     # Smaller vector size
    window=3,           # Small context window
    min_count=2,        # Ignore rare words
    workers=4,          # Number of CPU cores to use
    sg=1,               # Skip-Gram (better for small corpora)
    epochs=100,         # More epochs for small data
    sample=1e-5         # Subsampling of frequent words
)


In [ ]:
model.wv.most_similar("her", topn=20)

[('his', 0.9329653382301331),
 ('.', 0.8827051520347595),
 ('and', 0.8278223276138306),
 ('she', 0.8220826387405396),
 ('that', 0.8203333020210266),
 ('a', 0.814626932144165),
 ('in', 0.8142633438110352),
 ('to', 0.8101786971092224),
 ('throughout', 0.8062559366226196),
 ('with', 0.8056733012199402),
 (',', 0.7967996597290039),
 ('he', 0.7885398864746094),
 ("'s", 0.7862935066223145),
 ('work', 0.7809227705001831),
 ('through', 0.780012845993042),
 ('identifier', 0.7783862948417664),
 ('has', 0.7762808203697205),
 ('clinical', 0.7745503783226013),
 ('which', 0.7697321176528931),
 ('an', 0.7643749713897705)]

In [ ]:
# Initial gendered word list
gendered_words = ["madam"]

# Find related words in Word2Vec space
related_words = set(gendered_words)
for word in gendered_words:
    try:
        similar_words = model.wv.most_similar(word, topn=10)
        related_words.update([w[0] for w in similar_words])
    except KeyError:
        continue

print("Expanded gendered words:", related_words)

Expanded gendered words: {'bottle', 'glottis', 'madam', 'palate', 'telephone', 'fentanyl', 'https', 'overconfidence', 'sir', 'homemade', 'chemicals'}


In [ ]:
# # Load pre-trained Google Word2Vec model
# model = api.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
model.most_similar("brother", topn=50)

[('younger_brother', 0.8773985505104065),
 ('son', 0.8379532098770142),
 ('nephew', 0.8305360078811646),
 ('father', 0.824080765247345),
 ('uncle', 0.8177263140678406),
 ('cousin', 0.7986825108528137),
 ('brothers', 0.7532903552055359),
 ('twin_brother', 0.7497867345809937),
 ('eldest_son', 0.7214021682739258),
 ('elder_brother', 0.7205877304077148),
 ('sister', 0.7160382866859436),
 ('sons', 0.7071635127067566),
 ('dad', 0.6992479562759399),
 ('stepfather', 0.6958680748939514),
 ('grandson', 0.6951757669448853),
 ('stepson', 0.6736763119697571),
 ('siblings', 0.6570252776145935),
 ('stepbrother', 0.6560764908790588),
 ('daughter', 0.6544503569602966),
 ('grandfather', 0.65240877866745),
 ('nephews', 0.6438215970993042),
 ('husband', 0.6417980194091797),
 ('grandsons', 0.6386956572532654),
 ('niece', 0.6312658786773682),
 ('mother', 0.6273878812789917),
 ('uncles', 0.6240216493606567),
 ('aunt', 0.6217002272605896),
 ('borther', 0.6170939803123474),
 ('stepdad', 0.6142634153366089),
 (